# VASP to VESTA Magnetic Structure Visualizer

A modular, object-oriented Python tool designed to convert VASP calculation outputs (`CONTCAR` and `OUTCAR`) into fully formatted `.vesta` visualization files.

This tool automates the extraction of final-step atomic magnetic moments, scales magnetization vectors proportionally to physical moments, draws atomic bonds, adjusts atom sphere sizes directly inside VESTA, and applies correct crystallographic basis transformations.

---

## Features

* **Collinear and Non-Collinear Magnetism:** Automatically parses collinear spin calculations (`ISPIN = 2`, scalar moments projected along $z$) and non-collinear calculations (`LNONCOLLINEAR = .TRUE.`, 3D vector components $m_x, m_y, m_z$).
* **Crystallographic Basis Transformation:** Transforms Cartesian magnetic moments into VESTA’s internal unit vector coordinate frame $(\hat{\mathbf{a}}, \hat{\mathbf{b}}, \hat{\mathbf{c}})$, preserving the correct physical vector orientation across non-orthogonal crystal systems (hexagonal, monoclinic, triclinic).
* **Proportional Vector Scaling:** Scales vector lengths proportionally relative to the largest magnetic moment in the unit cell rather than forcing uniform arrow sizes.
* **Custom Atom Ball Radii:** Injects explicit `SITET` and `ATOMT` tables to override VESTA's hardcoded atom radii, allowing atoms to be scaled down (e.g., 50% or smaller) so vectors and bonding networks are unobstructed.
* **Color Preservation:** Preserves VESTA-style element colors (such as metallic grey for Nickel, Palladium, and Platinum) and provides a direct color-override dictionary.
* **Automatic Bond Generation:** Automatically computes bond search boundaries using Cordero covalent radii with configurable tolerance factors.
* **Spin-Polarized Vector Coloring:** Colors magnetic vectors by the sign of their $m_z$ component (Red for spin-up / $+m_z$, Blue for spin-down / $-m_z$), or uses a custom uniform palette.
* **In-Notebook Configuration:** Fully parameter-driven at the top of the file—no command-line flags required.

---

## Prerequisites

### Python Dependencies

* Python 3.8+
* `numpy`

Install dependencies via pip:

```bash
pip install numpy

```

### VASP Calculation Requirements

To extract atomic magnetic moments, your VASP calculation (`INCAR`) must output local site magnetizations:

* `LORBIT = 10` or `LORBIT = 11` (RWIGS must be specified if `LORBIT < 10`).
* `ISPIN = 2` (collinear spin-polarization) OR `LNONCOLLINEAR = .TRUE.` (non-collinear spin-orbit / magnetic calculation).

Both the relaxed coordinate file (`CONTCAR` or `POSCAR` in VASP 5+ format) and the output file (`OUTCAR`) must reside in the working directory.

---

## Object-Oriented Architecture

The script follows strict separation of concerns through an object-oriented design:

```
├── AtomSite                    # Dataclass for atomic positions, labels, and magnetic moments
├── ElementProperty             # Stores elemental metadata (Z, baseline covalent radius, RGB)
├── BondSpecification          # Dataclass defining pairwise bond parameters for VESTA's SBOND
├── CrystalStructure           # Manages unit cell geometry, cell metric conversions, and sites
├── AtomPropertyManager         # Manages element palettes, standard radii, and custom overrides
├── BondGenerator               # Evaluates covalent radii to generate pairwise bonding rules
├── ContcarParser               # Reads and normalizes VASP 5+ structural files
├── OutcarMagnetizationParser   # Extracts final ionic step magnetic moments
├── MagnetizationProcessor      # Filters cutoffs, scales magnitudes, and projects vectors
└── VestaWriter                 # Serializes structure and styling into the .vesta file format

```

---

## Configuration

All parameters are configured in the top section of the script or Jupyter Notebook cell:

In [ ]:
# ==============================================================================
# CONFIGURATION SETTINGS
# ==============================================================================
CONTCAR_PATH: str = "CONTCAR"          # Input VASP structure file (or POSCAR)
OUTCAR_PATH: str = "OUTCAR"            # Input VASP output file with magnetization
OUTPUT_VESTA_PATH: str = "magmom.vesta"# Output VESTA file

# Sizing Adjustments
ATOM_RADIUS_SCALE: float = 0.50        # Atom radius scaling factor (0.50 = 2x smaller)
BOND_RADIUS: float = 0.075             # Bond cylinder radius in Angstroms

# Custom Element Color Overrides (RGB 0-255)
CUSTOM_ELEMENT_COLORS: Dict[str, Tuple[int, int, int]] = {
    "Ni": (165, 165, 165),  # Enforce metallic grey instead of CPK green
    "Pd": (165, 165, 165),
    "Pt": (165, 165, 165),
}

# Magnetization Vector Appearance
MAG_CUTOFF: float = 0.05               # Min |M| (in mu_B) to display an arrow
MAX_ARROW_LENGTH: float = 0.90         # Length (in Angstroms) of the largest moment
ARROW_RADIUS: float = 0.20             # Shaft thickness of the vector cylinder (in Angstroms)
COLOR_BY_SPIN_Z: bool = True           # True: Red (+mz / spin-up), Blue (-mz / spin-down)
DEFAULT_COLOR: Tuple[int, int, int] = (255, 40, 40)  # RGB if COLOR_BY_SPIN_Z is False

# Bond Detection Settings
AUTO_GENERATE_BONDS: bool = True       # Automatically detect and draw crystal bonds
BOND_TOLERANCE_FACTOR: float = 1.25    # Multiplier: d_max = factor * (r_cov1 + r_cov2)
MANUAL_BONDS: Optional[List[Tuple[str, str, float, float]]] = None

# Atom Filtering (Set to None to include all magnetic atoms above cutoff)
TARGET_ELEMENTS: Optional[List[str]] = None  # e.g., ["Fe", "Mn", "Cr"]
TARGET_INDICES: Optional[List[int]] = None   # e.g., [1, 2, 5, 6] (1-based indices)

---

## Usage

### In VSCode / Jupyter Notebook (`.ipynb`)

1. Place your `CONTCAR` and `OUTCAR` in the same directory as your notebook.
2. Paste the script into a notebook cell.
3. Adjust any variables in the configuration block if needed.
4. Execute the cell (`Shift + Enter`).
5. Open the generated `magmom.vesta` file directly in VESTA.

### In Terminal / Standalone Script (`.py`)

```bash
python vasp_to_vesta.py

```

Upon execution, a summary table displays all rendered sites, extracted Cartesian magnetic moments, total magnitudes, and scaled visual arrow lengths:

```text
Generated 'magmom.vesta' successfully.
- Atom balls shrunk by factor of 2 (scale = 50% of default radii):
    * Ni: 0.625 Å | RGB: (165, 165, 165)
    * O:  0.370 Å | RGB: (255, 13, 13)
- Bond thickness (radius): 0.075 Å
- Vector cylinder radius: 0.20 Å | Max arrow length: 0.90 Å
---------------------------------------------------------------------------
Index   Label    Raw (mx, my, mz) [mu_B]        |M| [mu_B]   Arrow Length
---------------------------------------------------------------------------
1       Ni1      ( 0.000,  0.000,  1.721)        1.721        0.900 Å
2       Ni2      ( 0.000,  0.000, -1.718)        1.718        0.898 Å

```

---

## Technical Details

### 1. Vector Projection

VESTA interprets vector components in the `VECTR` block along the directions of the crystallographic axes $(\mathbf{a}, \mathbf{b}, \mathbf{c})$ normalized by the unit cell parameters:

$$\mathbf{v}_{\text{vesta}} = \left( \mathbf{v}_{\text{cart}} \cdot \mathbf{A}^{-1} \right) \odot \begin{bmatrix} a \\ b \\ c \end{bmatrix}$$

where $\mathbf{A}$ is the matrix whose rows are the direct lattice vectors, and $\odot$ denotes element-wise multiplication. This transformation prevents vector distortion in non-cubic lattices.

### 2. VESTA Sphere Sizing

VESTA ignores standard scale multipliers if the internal site table (`SITET`) and atom table (`ATOMT`) are missing, falling back to 100% covalent/van der Waals radii. The script generates explicit `SITET` and `ATOMT` blocks containing customized radius entries ($r_{\text{scaled}} = r_{\text{baseline}} \times \text{ATOM\_RADIUS\_SCALE}$) to guarantee that atom balls shrink to the desired visual size.

---

## Troubleshooting

| Issue | Cause | Solution |
| --- | --- | --- |
| `ValueError: No 'magnetization (x)' block matching N atoms was found` | `LORBIT` was omitted or set below 10 in `INCAR`. | Re-run VASP with `LORBIT = 11` (or `LORBIT = 10`). |
| `ValueError: VASP 4 format detected` | Line 6 of `CONTCAR` contains numbers instead of elemental species. | Add the element symbols to line 6 of the `CONTCAR` / `POSCAR` file. |
| Vectors do not appear in VESTA | Moment magnitudes are below `MAG_CUTOFF`. | Lower `MAG_CUTOFF` (e.g., to `0.01`). |
| Vectors are pointed along $z$ only | Calculation was run with collinear spin (`ISPIN = 2`). | Standard collinear calculations only compute scalar spin projections. Run with `LNONCOLLINEAR = .TRUE.` for full 3D vector directions. |

---

## License

This project is open-source and available under the [MIT License](https://www.google.com/search?q=LICENSE).